In [0]:
%run /Repos/awproject/databricks_learning/config/config


In [0]:


%run /Repos/awproject/databricks_learning/batch_control/batch_control



In [0]:

from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable


# =============================================================================
# CELL 4 — helpers
# =============================================================================

def add_audit_cols(df):
    """Drop silver audit cols and stamp gold audit cols."""
    return (
        df
        .drop("silver_ingestion_timestamp", "silver_ingestion_date",
              "dq_status", "dq_failed_reason")
        .withColumn("gold_ingestion_timestamp", F.current_timestamp())
        .withColumn("gold_ingestion_date",      F.current_date())
    )


def generate_surrogate_key(df, nk_col: str, sk_col: str):
    """
    Generate a surrogate integer key based on row number.
    Used for first-time load of SCD1 dims.
    """
    return df.withColumn(sk_col,
        F.row_number().over(Window.orderBy(nk_col)).cast("long")
    )


def write_gold_full(df, table_name: str):
    """SCD1 and static dims — full overwrite."""
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD}.{table_name}")
    )
    rows = df.count()
    print(f"  Written {rows:,} rows → {GOLD}.{table_name}")
    return rows


def scd2_merge(df_new, table_name: str, nk_col: str, sk_col: str, track_cols: list):
    """
    SCD Type 2 merge:
      - New records   → insert with is_current = True
      - Changed rows  → expire old (is_current = False, expiry_date = today)
                      → insert new version (is_current = True)
      - Unchanged     → no action
    """
    target_table = f"{GOLD}.{table_name}"

    # Add hash of tracked columns to detect changes
    df_new = df_new.withColumn("_row_hash",
        F.md5(F.concat_ws("|", *[F.col(c).cast("string") for c in track_cols]))
    )

    if not spark.catalog.tableExists(target_table):
        # First load — insert all as current
        df_new = df_new \
            .withColumn("effective_date", F.current_date()) \
            .withColumn("expiry_date",    F.lit("9999-12-31").cast("date")) \
            .withColumn("is_current",     F.lit(True)) \
            .drop("_row_hash")
        write_gold_full(df_new, table_name)
        return df_new.count()

    dt = DeltaTable.forName(spark, target_table)

    # Step 1 — expire changed rows
    dt.alias("t").merge(
        df_new.alias("s"),
        f"t.{nk_col} = s.{nk_col} AND t.is_current = true AND t._row_hash != s._row_hash"
    ).whenMatchedUpdate(set={
        "is_current":   "false",
        "expiry_date":  "current_date()"
    }).execute()

    # Step 2 — insert new / changed rows (not matched on nk + is_current + same hash)
    df_insert = df_new \
        .withColumn("effective_date", F.current_date()) \
        .withColumn("expiry_date",    F.lit("9999-12-31").cast("date")) \
        .withColumn("is_current",     F.lit(True)) \
        .drop("_row_hash")

    dt.alias("t").merge(
        df_insert.alias("s"),
        f"t.{nk_col} = s.{nk_col} AND t.is_current = true"
    ).whenNotMatchedInsertAll() \
     .execute()

    total = spark.read.table(target_table).filter(F.col("is_current") == True).count()
    print(f"  SCD2 complete → {target_table} | {total:,} current rows")
    return total


# =============================================================================
# CELL 5 — dim_date (static — generated, not from silver)
# Covers 2015-01-01 to 2018-12-31 based on sales data range
# =============================================================================
table_name  = "dim_date"
ctrl_silver = get_last_load("calendar", "silver")
ctrl_gold   = get_last_load(table_name, "gold")

if ctrl_silver["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — silver calendar status: {ctrl_silver['status']}")
elif ctrl_gold["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_gold['load_type']}] {GOLD}.{table_name}")
    log_batch_start(table_name, "gold", ctrl_gold["load_type"])
    try:
        df_date = spark.range(0, 365 * 4).select(
            F.expr("date_add(date'2015-01-01', cast(id as int))").alias("full_date")
        ).select(
            F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
            F.col("full_date"),
            F.year("full_date").alias("year"),
            F.quarter("full_date").alias("quarter"),
            F.month("full_date").alias("month"),
            F.date_format("full_date", "MMMM").alias("month_name"),
            F.date_format("full_date", "MMM").alias("month_short"),
            F.weekofyear("full_date").alias("week_of_year"),
            F.dayofmonth("full_date").alias("day_of_month"),
            F.dayofweek("full_date").alias("day_of_week"),
            F.date_format("full_date", "EEEE").alias("day_name"),
            F.when(F.dayofweek("full_date").isin(1, 7), True)
             .otherwise(False).alias("is_weekend"),
        ) \
        .withColumn("gold_ingestion_timestamp", F.current_timestamp()) \
        .withColumn("gold_ingestion_date",      F.current_date())

        rows = write_gold_full(df_date, table_name)
        log_batch_end(table_name, "gold", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "gold", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 6 — dim_territory (SCD1)
# Source  : silver.territories
# SK      : territory_sk
# NK      : sales_territory_key
# =============================================================================
table_name  = "dim_territory"
ctrl_silver = get_last_load("territories", "silver")
ctrl_gold   = get_last_load(table_name, "gold")

if ctrl_silver["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — silver territories status: {ctrl_silver['status']}")
elif ctrl_gold["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_gold['load_type']}] {GOLD}.{table_name}")
    log_batch_start(table_name, "gold", ctrl_gold["load_type"])
    try:
        df = spark.read.table(f"{SILVER}.territories") \
            .filter(F.col("dq_status") == "PASS") \
            .select("sales_territory_key", "region", "country", "continent")

        # Surrogate key
        df = generate_surrogate_key(df, "sales_territory_key", "territory_sk")

        df = df.select(
            "territory_sk",
            "sales_territory_key",
            "region", "country", "continent"
        )

        df   = add_audit_cols(df)
        rows = write_gold_full(df, table_name)
        log_batch_end(table_name, "gold", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "gold", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 7 — dim_product_category (SCD1 — snowflake node)
# Source  : silver.product_categories + silver.product_subcategories
# SK      : product_category_sk
# NK      : product_subcategory_key (joined with category)
# =============================================================================
table_name  = "dim_product_category"
ctrl_silver = get_last_load("product_categories", "silver")
ctrl_gold   = get_last_load(table_name, "gold")

if ctrl_silver["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — silver product_categories status: {ctrl_silver['status']}")
elif ctrl_gold["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_gold['load_type']}] {GOLD}.{table_name}")
    log_batch_start(table_name, "gold", ctrl_gold["load_type"])
    try:
        df_cat = spark.read.table(f"{SILVER}.product_categories") \
            .filter(F.col("dq_status") == "PASS") \
            .select("product_category_key", "category_name")

        df_sub = spark.read.table(f"{SILVER}.product_subcategories") \
            .filter(F.col("dq_status") == "PASS") \
            .select("product_subcategory_key", "subcategory_name", "product_category_key")

        # Join subcategory with category
        df = df_sub.join(df_cat, "product_category_key", "left")

        # Surrogate key on subcategory
        df = generate_surrogate_key(df, "product_subcategory_key", "product_category_sk")

        df = df.select(
            "product_category_sk",
            "product_subcategory_key",
            "subcategory_name",
            "product_category_key",
            "category_name"
        )

        df   = add_audit_cols(df)
        rows = write_gold_full(df, table_name)
        log_batch_end(table_name, "gold", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "gold", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 8 — dim_product (SCD2)
# Source  : silver.products joined with dim_product_category
# SK      : product_sk
# NK      : product_key
# Track   : product_name, product_cost, product_price, product_color
# =============================================================================
table_name  = "dim_product"
ctrl_silver = get_last_load("products", "silver")
ctrl_gold   = get_last_load(table_name, "gold")

if ctrl_silver["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — silver products status: {ctrl_silver['status']}")
elif ctrl_gold["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_gold['load_type']}] {GOLD}.{table_name}")
    log_batch_start(table_name, "gold", ctrl_gold["load_type"])
    try:
        df_prod = spark.read.table(f"{SILVER}.products") \
            .filter(F.col("dq_status") == "PASS") \
            .select("product_key", "product_subcategory_key", "product_sku",
                    "product_name", "model_name", "product_color",
                    "product_size", "product_style",
                    "product_cost", "product_price")

        # Join to get product_category_sk from gold dim
        df_cat = spark.read.table(f"{GOLD}.dim_product_category") \
            .select("product_category_sk", "product_subcategory_key")

        df = df_prod.join(df_cat, "product_subcategory_key", "left")

        # Surrogate key
        df = generate_surrogate_key(df, "product_key", "product_sk")

        df = df.select(
            "product_sk",
            "product_key",
            "product_sku",
            "product_name",
            "model_name",
            "product_color",
            "product_size",
            "product_style",
            "product_cost",
            "product_price",
            "product_category_sk",
        )

        rows = scd2_merge(
            df_new      = add_audit_cols(df),
            table_name  = table_name,
            nk_col      = "product_key",
            sk_col      = "product_sk",
            track_cols  = ["product_name", "product_cost",
                           "product_price", "product_color"]
        )
        log_batch_end(table_name, "gold", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "gold", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 9 — dim_customer (SCD2)
# Source  : silver.customers
# SK      : customer_sk
# NK      : customer_key
# Track   : annual_income, occupation, education_level, marital_status
# =============================================================================
table_name  = "dim_customer"
ctrl_silver = get_last_load("customers", "silver")
ctrl_gold   = get_last_load(table_name, "gold")

if ctrl_silver["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — silver customers status: {ctrl_silver['status']}")
elif ctrl_gold["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_gold['load_type']}] {GOLD}.{table_name}")
    log_batch_start(table_name, "gold", ctrl_gold["load_type"])
    try:
        df = spark.read.table(f"{SILVER}.customers") \
            .filter(F.col("dq_status") == "PASS") \
            .select("customer_key", "prefix", "first_name", "last_name",
                    "birth_date", "marital_status", "gender",
                    "email_address", "annual_income", "total_children",
                    "education_level", "occupation", "home_owner")

        # Full name derived column
        df = df.withColumn("full_name",
            F.concat_ws(" ", F.col("prefix"), F.col("first_name"), F.col("last_name"))
        )

        # Surrogate key
        df = generate_surrogate_key(df, "customer_key", "customer_sk")

        df = df.select(
            "customer_sk",
            "customer_key",
            "full_name",
            "first_name",
            "last_name",
            "prefix",
            "gender",
            "birth_date",
            "marital_status",
            "email_address",
            "annual_income",
            "total_children",
            "education_level",
            "occupation",
            "home_owner",
        )

        rows = scd2_merge(
            df_new      = add_audit_cols(df),
            table_name  = table_name,
            nk_col      = "customer_key",
            sk_col      = "customer_sk",
            track_cols  = ["annual_income", "occupation",
                           "education_level", "marital_status"]
        )
        log_batch_end(table_name, "gold", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "gold", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 10 — fact_sales
# Source  : silver.sales (PASS rows only)
# Joins   : dim_customer, dim_product, dim_territory, dim_date (surrogate keys)
# Grain   : one row per order_number + order_line_item
# =============================================================================
table_name  = "fact_sales"
ctrl_silver = get_last_load("sales", "silver")
ctrl_gold   = get_last_load(table_name, "gold")

if ctrl_silver["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — silver sales status: {ctrl_silver['status']}")
elif ctrl_gold["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_gold['load_type']}] {GOLD}.{table_name}")
    log_batch_start(table_name, "gold", ctrl_gold["load_type"])
    try:
        df_sales = spark.read.table(f"{SILVER}.sales") \
            .filter(F.col("dq_status") == "PASS")

        # Lookup surrogate keys from dims — use current rows only for SCD2
        df_cust = spark.read.table(f"{GOLD}.dim_customer") \
            .filter(F.col("is_current") == True) \
            .select(
                F.col("customer_sk"),
                F.col("customer_key").alias("customer_key_lkp")
            )

        df_prod = spark.read.table(f"{GOLD}.dim_product") \
            .filter(F.col("is_current") == True) \
            .select(
                F.col("product_sk"),
                F.col("product_key").alias("product_key_lkp")
            )

        df_terr = spark.read.table(f"{GOLD}.dim_territory") \
            .select(
                F.col("territory_sk"),
                F.col("sales_territory_key").alias("territory_key_lkp")
            )

        # date_key from dim_date — format order_date as YYYYMMDD int
        df_fact = df_sales \
            .withColumn("date_key",
                F.date_format(F.col("order_date"), "yyyyMMdd").cast("int")
            ) \
            .join(df_cust, df_sales.customer_key  == df_cust.customer_key_lkp,  "left") \
            .join(df_prod, df_sales.product_key   == df_prod.product_key_lkp,   "left") \
            .join(df_terr, df_sales.territory_key == df_terr.territory_key_lkp, "left") \
            .select(
                # Surrogate keys
                F.col("customer_sk"),
                F.col("product_sk"),
                F.col("territory_sk"),
                F.col("date_key"),
                # Degenerate dimensions
                F.col("order_number"),
                F.col("order_line_item"),
                # Measures
                F.col("order_date"),
                F.col("stock_date"),
                F.col("order_quantity"),
                # Audit
                F.current_timestamp().alias("gold_ingestion_timestamp"),
                F.current_date().alias("gold_ingestion_date"),
            )

        rows = write_gold_full(df_fact, table_name)
        log_batch_end(table_name, "gold", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "gold", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 11 — summary
# =============================================================================
show_control_table("gold")

print("\n=== GOLD TABLE ROW COUNTS ===")
for t in ["dim_date", "dim_territory", "dim_product_category",
          "dim_product", "dim_customer", "fact_sales"]:
    try:
        df  = spark.read.table(f"{GOLD}.{t}")
        cnt = df.count()
        # For SCD2 dims show current vs total
        if "is_current" in df.columns:
            curr = df.filter(F.col("is_current") == True).count()
            print(f"  {t:<25} {cnt:>8,} total | {curr:,} current")
        else:
            print(f"  {t:<25} {cnt:>8,} rows")
    except Exception:
        print(f"  {t:<25} table not found")